# 1. Installation & Imports
This cell sets up the environment for Kaggle T4 dual-GPU speculative decoding.

In [ ]:
!pip install transformers accelerate

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# 2. Prompts Dataset
Our 50-prompt test suite spanning factual, long-form, code, and chat tasks.

In [ ]:
TEST_PROMPTS = [
    # Factual (15 prompts)
    "What is the capital of France?",
    "Who wrote the play Hamlet?",
    "What is the chemical symbol for gold?",
    "When did the first moon landing occur?",
    "What is the largest planet in our solar system?",
    "Who painted the Mona Lisa?",
    "What is the speed of light in a vacuum?",
    "How many continents are there on Earth?",
    "What is the boiling point of water at sea level?",
    "Who was the first President of the United States?",
    "What is the deepest ocean trench?",
    "What gas do plants absorb from the atmosphere?",
    "Who discovered penicillin?",
    "What year did World War II end?",
    "What is the tallest mountain in the world?",

    # Long-form / Explanatory (15 prompts)
    "Explain photosynthesis in detail, breaking down the light-dependent and light-independent reactions.",
    "Describe the causes and consequences of the French Revolution.",
    "How does a quantum computer differ from a classical computer?",
    "What are the main differences between meiosis and mitosis?",
    "Write a short essay on the impact of artificial intelligence on modern healthcare.",
    "Explain the theory of general relativity in simple terms.",
    "Discuss the evolution of the internet from ARPANET to the modern web.",
    "What is the greenhouse effect and how does it contribute to climate change?",
    "Describe the water cycle and its importance to Earth's ecosystems.",
    "Explain how the human immune system fights off a viral infection.",
    "Compare and contrast capitalism and socialism as economic systems.",
    "What are black holes, and how do they form?",
    "Describe the process of natural selection with an example.",
    "Explain how a combustion engine works.",
    "What is blockchain technology and how does it secure transactions?",

    # Code-like / Structured (10 prompts)
    "Write a Python function to compute the nth Fibonacci number using dynamic programming.",
    "How do you reverse a linked list in C++?",
    "Write an SQL query to find the second highest salary from an Employee table.",
    "Explain the concept of closures in JavaScript with an example.",
    "Show how to implement a binary search tree insertion in Java.",
    "Write a bash script to find all files larger than 100MB in a directory.",
    "How do you implement a singleton pattern in Python?",
    "Write a Go function to concurrently fetch three URLs using goroutines.",
    "Provide a basic Dockerfile for a Node.js web application.",
    "Explain the difference between a process and a thread, with code examples.",

    # Multi-turn style / Instruct (10 prompts)
    "User: I need a recipe for chocolate chip cookies. Assistant: Here is a recipe... User: Can you make it vegan?",
    "User: Hello, how are you? Assistant: I am an AI, I don't have feelings, but I'm here to help! User: Tell me a joke.",
    "User: Translate 'hello world' to French. Assistant: Bonjour le monde. User: Now to Spanish.",
    "User: What is 2 + 2? Assistant: 4. User: Multiply that by 5.",
    "User: Name a color. Assistant: Blue. User: Give me three things that are that color.",
    "User: Write a poem about a cat. Assistant: [Poem] User: Now make it rhyme.",
    "User: What's the best way to invest $1000? Assistant: [Advice] User: What about crypto?",
    "User: Give me a workout routine. Assistant: [Routine] User: I don't have weights.",
    "User: Summarize the plot of Inception. Assistant: [Summary] User: Explain the ending.",
    "User: How do I boil an egg? Assistant: [Steps] User: What if I want it soft-boiled?"
]

# 3. Model Loading & Baseline Generator
Functions to load the Target (7B) and Draft (0.5B) models in FP16 on separate GPUs, plus the standard autoregressive baseline.

In [ ]:
def load_models_and_tokenizer(
    draft_model_id="Qwen/Qwen2.5-0.5B-Instruct",
    target_model_id="Qwen/Qwen2.5-7B-Instruct",
    target_device="cuda:0",
    draft_device="cuda:1" # Phase 4 target
):
    dtype = torch.float16 # Required to fit 7B on Kaggle 16GB T4 GPUs

    # Fallback to single GPU for local syntax testing if 2 GPUs aren't available
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        print("Less than 2 GPUs found. Defaulting both models to cuda:0 for safety.")
        draft_device = "cuda:0" if torch.cuda.is_available() else "cpu"
        target_device = draft_device

    print(f"Loading tokenizer from {target_model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(target_model_id)
    
    print(f"Loading tokenizer from {draft_model_id} for verification...")
    draft_tokenizer = AutoTokenizer.from_pretrained(draft_model_id)
    
    print("Asserting tokenizer vocabularies are identical...")
    assert draft_tokenizer.get_vocab() == tokenizer.get_vocab(), \
        "CRITICAL ERROR: Draft and target models do not share the exact same tokenizer vocabulary!"
        
    print(f"Loading draft model ({draft_model_id}) on {draft_device} in {dtype}...")
    draft_model = AutoModelForCausalLM.from_pretrained(
        draft_model_id,
        torch_dtype=dtype,
        device_map=draft_device
    )
    draft_model.eval()

    print(f"Loading target model ({target_model_id}) on {target_device} in {dtype}...")
    target_model = AutoModelForCausalLM.from_pretrained(
        target_model_id,
        torch_dtype=dtype,
        device_map=target_device
    )
    target_model.eval()
    
    return draft_model, target_model, tokenizer

def naive_greedy_generate(model, tokenizer, input_ids, max_new_tokens):
    with torch.no_grad():
        outputs = model(input_ids=input_ids, use_cache=True)
        past = outputs.past_key_values
        next_token = torch.argmax(
            outputs.logits[:, -1, :].float(), dim=-1
        ).unsqueeze(-1)

        generated = [input_ids, next_token]

        for _ in range(max_new_tokens - 1):
            outputs = model(
                input_ids=next_token,
                past_key_values=past,
                use_cache=True,
            )
            past = outputs.past_key_values
            next_token = torch.argmax(
                outputs.logits[:, -1, :].float(), dim=-1
            ).unsqueeze(-1)
            generated.append(next_token)

            if next_token[0, 0].item() == tokenizer.eos_token_id:
                break

    return torch.cat(generated, dim=1)

# 4. The Speculative Decoding Engine
The core engine implementing KV-cache rollback, log-space acceptance, and residual resampling mathematically.

In [ ]:
class SpeculativeDecoder:
    def __init__(self, draft_model, target_model, tokenizer, k=4):
        self.draft_model = draft_model
        self.target_model = target_model
        self.tokenizer = tokenizer
        self.k = k
        self.draft_device = next(draft_model.parameters()).device
        self.target_device = next(target_model.parameters()).device
        
        # Profiling stats
        self.transfer_time_ms = 0.0
        self.compute_time_ms = 0.0
        
        # Alpha tracking stats
        self.total_draft_tokens_proposed = 0
        self.total_draft_tokens_accepted = 0
        
    def reset_profiler(self):
        self.transfer_time_ms = 0.0
        self.compute_time_ms = 0.0
        self.total_draft_tokens_proposed = 0
        self.total_draft_tokens_accepted = 0

    def _transfer(self, tensor, device):
        """Cross-device tensor transfer with strict event isolation."""
        if str(tensor.device) == str(device):
            return tensor
            
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        res = tensor.to(device)
        end.record()
        torch.cuda.synchronize()
        self.transfer_time_ms += start.elapsed_time(end)
        return res
        
    def _run_compute(self, fn, *args, **kwargs):
        """Wraps compute functions to track compute isolation time."""
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        res = fn(*args, **kwargs)
        end.record()
        torch.cuda.synchronize()
        self.compute_time_ms += start.elapsed_time(end)
        return res

    def truncate_kv_cache(self, past_key_values, keep_length):
        if past_key_values is None:
            return None
        past_key_values.crop(keep_length)
        for attr in ("_seen_tokens", "seen_tokens"):
            if hasattr(past_key_values, attr):
                current = getattr(past_key_values, attr)
                if isinstance(current, int) and current != keep_length:
                    setattr(past_key_values, attr, keep_length)
        actual = past_key_values.get_seq_length()
        assert actual == keep_length, (
            f"After crop({keep_length}), get_seq_length()={actual}. "
            f"Internal state: {past_key_values.__dict__}"
        )
        return past_key_values

    def draft_k_tokens(self, input_ids, past_key_values, temperature=0.0):
        draft_tokens = []
        draft_logits_list = []
        current_input_ids = input_ids

        for _ in range(self.k):
            with torch.no_grad():
                outputs = self.draft_model(
                    input_ids=current_input_ids,
                    past_key_values=past_key_values,
                    use_cache=True,
                )
            past_key_values = outputs.past_key_values
            logits_f32 = outputs.logits[:, -1, :].float()
            draft_logits_list.append(logits_f32)
            
            if temperature == 0.0:
                next_token_id = torch.argmax(logits_f32, dim=-1).unsqueeze(-1)
            else:
                probs = F.softmax(logits_f32 / temperature, dim=-1)
                next_token_id = torch.multinomial(probs, num_samples=1)
                
            draft_tokens.append(next_token_id)
            current_input_ids = next_token_id

        return (
            torch.cat(draft_tokens, dim=1),
            torch.stack(draft_logits_list, dim=1),
            past_key_values,
        )

    def verify_with_target(self, input_ids, past_key_values):
        with torch.no_grad():
            outputs = self.target_model(
                input_ids=input_ids,
                past_key_values=past_key_values,
                use_cache=True,
            )
        return outputs.logits.float(), outputs.past_key_values

    def accept_or_reject_greedy(self, target_logits, draft_tokens):
        batch_size, k = draft_tokens.shape
        assert batch_size == 1, "Only batch_size=1 supported for now"

        target_predictions = torch.argmax(target_logits, dim=-1)

        accepted_tokens = []
        num_accepted = 0

        for i in range(k):
            if draft_tokens[0, i].item() == target_predictions[0, i].item():
                accepted_tokens.append(draft_tokens[0, i].item())
                num_accepted += 1
            else:
                break

        replacement_token = target_predictions[:, num_accepted].unsqueeze(-1)

        if accepted_tokens:
            accepted_tensor = torch.tensor(
                accepted_tokens, device=draft_tokens.device
            ).unsqueeze(0)
        else:
            accepted_tensor = torch.empty(
                (1, 0), dtype=torch.long, device=draft_tokens.device
            )

        return accepted_tensor, replacement_token, num_accepted

    def accept_or_reject_sampling(self, target_logits, draft_logits, draft_tokens, temperature):
        batch_size, k = draft_tokens.shape
        assert batch_size == 1, "Only batch_size=1 supported for now"

        target_log_probs = F.log_softmax(target_logits / temperature, dim=-1)
        draft_log_probs = F.log_softmax(draft_logits / temperature, dim=-1)

        accepted_tokens = []
        num_accepted = 0

        for i in range(k):
            token = draft_tokens[0, i].item()
            log_p_target = target_log_probs[0, i, token]
            log_p_draft = draft_log_probs[0, i, token]

            log_ratio = log_p_target - log_p_draft
            r = torch.rand(1, device=draft_tokens.device)
            log_r = torch.log(r).item()

            if log_r < log_ratio.item():
                accepted_tokens.append(token)
                num_accepted += 1
            else:
                break

        if num_accepted == k:
            bonus_probs = F.softmax(target_logits[:, k] / temperature, dim=-1)
            replacement_token = torch.multinomial(bonus_probs, num_samples=1)
        else:
            p_target = F.softmax(target_logits[:, num_accepted] / temperature, dim=-1)
            p_draft = F.softmax(draft_logits[:, num_accepted] / temperature, dim=-1)

            res = torch.clamp(p_target - p_draft, min=0.0)
            res_sum = res.sum(dim=-1, keepdim=True)

            if res_sum.item() > 1e-8:
                res_probs = res / res_sum
            else:
                res_probs = p_target

            replacement_token = torch.multinomial(res_probs, num_samples=1)

        if accepted_tokens:
            accepted_tensor = torch.tensor(
                accepted_tokens, device=draft_tokens.device
            ).unsqueeze(0)
        else:
            accepted_tensor = torch.empty(
                (1, 0), dtype=torch.long, device=draft_tokens.device
            )

        return accepted_tensor, replacement_token, num_accepted
        
    def _draft_step_compute(self, next_token, draft_past, temperature):
        """Isolated draft compute step."""
        with torch.no_grad():
            return self.draft_k_tokens(next_token, draft_past, temperature=temperature)
            
    def _target_step_compute(self, target_input, target_past):
        """Isolated target compute step."""
        with torch.no_grad():
            return self.verify_with_target(target_input, target_past)

    def speculative_generate(self, prompt_input_ids, max_new_tokens=50, temperature=0.0):
        current_input_ids = prompt_input_ids
        
        # Prefill: Pre-transfer inputs so transfer time is isolated from compute time
        draft_inputs = self._transfer(current_input_ids, self.draft_device)
        target_inputs = self._transfer(current_input_ids, self.target_device)
        
        start_compute = torch.cuda.Event(enable_timing=True)
        end_compute = torch.cuda.Event(enable_timing=True)
        
        start_compute.record()
        with torch.no_grad():
            draft_outputs = self.draft_model(input_ids=draft_inputs, use_cache=True)
            draft_past = draft_outputs.past_key_values

            target_outputs = self.target_model(input_ids=target_inputs, use_cache=True)
            target_past = target_outputs.past_key_values
            
            target_logits = target_outputs.logits[:, -1, :].float()
            if temperature == 0.0:
                next_token = torch.argmax(target_logits, dim=-1).unsqueeze(-1)
            else:
                next_token = torch.multinomial(F.softmax(target_logits / temperature, dim=-1), num_samples=1)

        end_compute.record()
        torch.cuda.synchronize()
        self.compute_time_ms += start_compute.elapsed_time(end_compute)

        generated_ids = [self._transfer(current_input_ids, self.target_device), next_token]
        num_generated = 1
        cache_len = current_input_ids.shape[1]

        while num_generated < max_new_tokens:
            # Transfer next token to draft
            next_token_draft = self._transfer(next_token, self.draft_device)
            
            # Draft K tokens
            draft_tokens, draft_logits, new_draft_past = self._run_compute(
                self._draft_step_compute, next_token_draft, draft_past, temperature
            )
            
            # Transfer drafts back to target
            draft_tokens_target = self._transfer(draft_tokens, self.target_device)
            target_input = torch.cat([next_token, draft_tokens_target], dim=1)
            
            # Target verification
            target_logits, new_target_past = self._run_compute(
                self._target_step_compute, target_input, target_past
            )
            
            # Transfer logits if sampling
            if temperature > 0.0:
                draft_logits_target = self._transfer(draft_logits, self.target_device)
            
            # Acceptance logic (purely on target device compute)
            start_compute.record()
            if temperature == 0.0:
                accepted_tokens, replacement_token, num_accepted = (
                    self.accept_or_reject_greedy(target_logits, draft_tokens_target)
                )
            else:
                accepted_tokens, replacement_token, num_accepted = (
                    self.accept_or_reject_sampling(target_logits, draft_logits_target, draft_tokens_target, temperature)
                )

            # Track stats
            self.total_draft_tokens_proposed += self.k
            self.total_draft_tokens_accepted += num_accepted

            if num_accepted > 0:
                generated_ids.append(accepted_tokens)
            generated_ids.append(replacement_token)
            num_generated += num_accepted + 1
            
            end_compute.record()
            torch.cuda.synchronize()
            self.compute_time_ms += start_compute.elapsed_time(end_compute)

            # KV cache rollback - Pre-transfer bonus token if all accepted
            if num_accepted == self.k:
                draft_bonus_input = self._transfer(draft_tokens_target[:, -1:], self.draft_device)
                
                start_compute.record()
                with torch.no_grad():
                    extra = self.draft_model(
                        input_ids=draft_bonus_input,
                        past_key_values=new_draft_past,
                        use_cache=True,
                    )
                new_draft_past = extra.past_key_values
                end_compute.record()
                torch.cuda.synchronize()
                self.compute_time_ms += start_compute.elapsed_time(end_compute)

            start_compute.record()
            keep_len = cache_len + 1 + num_accepted
            draft_past = self.truncate_kv_cache(new_draft_past, keep_len)
            target_past = self.truncate_kv_cache(new_target_past, keep_len)

            cache_len = keep_len
            next_token = replacement_token

            end_compute.record()
            torch.cuda.synchronize()
            self.compute_time_ms += start_compute.elapsed_time(end_compute)

            if next_token[0, 0].item() == self.tokenizer.eos_token_id:
                break

        return torch.cat(generated_ids, dim=1)

# 5. Verification & Benchmarking Harness
Test suite for Phase 3 (Math Proof), Phase 4 (Multi-GPU Profiling), and Phase 5 (K-Sweep Benchmarking).

In [ ]:
def test_phase_3_sampling_math_proof(decoder):
    print("\n--- Phase 3: Mathematical Proof of Residual Resampling ---")
    vocab_size = 10
    k = 1
    temperature = 1.0
    device = "cpu" if not torch.cuda.is_available() else "cuda:0"
    
    torch.manual_seed(42)
    draft_logits = torch.randn(1, k, vocab_size, device=device)
    target_logits = torch.randn(1, k + 1, vocab_size, device=device)
    p_target = F.softmax(target_logits[0, 0], dim=-1)
    
    N = 50000
    counts = torch.zeros(vocab_size, device=device)
    
    print(f"Running {N} iterations of purely the acceptance/rejection loop...")
    for _ in range(N):
        draft_probs = F.softmax(draft_logits[0, 0], dim=-1)
        draft_token = torch.multinomial(draft_probs, 1).unsqueeze(0)
        
        _, replacement, num_accepted = decoder.accept_or_reject_sampling(
            target_logits, draft_logits, draft_token, temperature
        )
        
        if num_accepted == 1:
            counts[draft_token.item()] += 1
        else:
            counts[replacement.item()] += 1
            
    p_empirical = (counts + 1e-6) / (N + 1e-6 * vocab_size)
    kl_div = F.kl_div(p_empirical.log(), p_target, reduction='sum').item()
    print(f"KL Divergence between Target p(x) and Empirical Spec-Decoding: {kl_div:.6f}")
    
    if kl_div < 0.01:
        print("PHASE 3 COMPLETE: Speculative Decoding rejection math is PERFECTLY ALIGNED!")
    else:
        print("WARNING: High KL Divergence. Mathematics failed.")

def test_phase_4_multi_gpu(decoder, target, tokenizer, device):
    print("\n--- Phase 4: Multi-GPU Verification & Profiling ---")
    if str(decoder.draft_device) == str(decoder.target_device):
        print("WARNING: Only 1 GPU found. Simulating transfer timing instead of true cross-device transfer.")
    else:
        print(f"Confirmed Multi-GPU Active: Draft on {decoder.draft_device}, Target on {decoder.target_device}")

    # Use first 10 prompts for quick profiling
    prompts_to_test = TEST_PROMPTS[:10]
    pass_count = 0
    max_new_tokens = 30
    
    decoder.reset_profiler()

    for i, prompt in enumerate(prompts_to_test):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = inputs.input_ids

        baseline_out = naive_greedy_generate(target, tokenizer, input_ids, max_new_tokens)
        
        with torch.no_grad():
            spec_out = decoder.speculative_generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.0
            )

        min_len = min(baseline_out.shape[1], spec_out.shape[1])
        baseline_trunc = baseline_out[0, :min_len]
        spec_trunc = spec_out[0, :min_len]

        if torch.equal(baseline_trunc, spec_trunc):
            pass_count += 1
        else:
            print(f"Phase 4 Failed exact match on prompt {i+1}")
            break

    print(f"Phase 4 Exact-Match Verification: {pass_count}/{len(prompts_to_test)} Passed.")
    
    print("\n--- Profiling Results (Over 10 Prompts) ---")
    print(f"Total Compute Time:  {decoder.compute_time_ms:.2f} ms")
    print(f"Total Transfer Time: {decoder.transfer_time_ms:.2f} ms")
    
    if decoder.compute_time_ms > 0:
        overhead_pct = (decoder.transfer_time_ms / decoder.compute_time_ms) * 100
        print(f"Transfer Overhead:   {overhead_pct:.2f}% of compute")
        
    if pass_count == len(prompts_to_test):
        print("PHASE 4 COMPLETE: Multi-GPU placement and verification is successful!")

def test_phase_5_benchmarking(decoder, target, draft, tokenizer, device):
    print("\n--- Phase 5: Benchmarking & K-Sweep ---")
    prompts = TEST_PROMPTS[:10]
    max_new_tokens = 100
    
    print("Measuring Baseline Target Model TPS...")
    start_time = time.time()
    total_tokens_baseline = 0
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        out = naive_greedy_generate(target, tokenizer, inputs.input_ids, max_new_tokens)
        total_tokens_baseline += (out.shape[1] - inputs.input_ids.shape[1])
    target_time = time.time() - start_time
    target_tps = total_tokens_baseline / target_time
    
    print("Measuring Baseline Draft Model TPS...")
    start_time = time.time()
    total_tokens_draft = 0
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(decoder.draft_device)
        out = naive_greedy_generate(draft, tokenizer, inputs.input_ids, max_new_tokens)
        total_tokens_draft += (out.shape[1] - inputs.input_ids.shape[1])
    draft_time = time.time() - start_time
    draft_tps = total_tokens_draft / draft_time
    
    c = target_tps / draft_tps  # Cost ratio: c_draft / c_target
    print(f"\nTarget TPS (c_target): {target_tps:.2f} tokens/sec")
    print(f"Draft TPS (c_draft):   {draft_tps:.2f} tokens/sec")
    print(f"Cost ratio c:          {c:.4f}")
    
    k_values = [1, 2, 4, 8, 16]
    
    print("\n| K  | Speedup | Tokens/sec | Acceptance (α) | Theo Opt K |")
    print("|----|---------|------------|----------------|------------|")
    
    for k in k_values:
        decoder.k = k
        decoder.reset_profiler()
        
        # Warmup (1 prompt)
        _ = decoder.speculative_generate(tokenizer(prompts[0], return_tensors="pt").input_ids.to(device), max_new_tokens=10, temperature=0.0)
        
        decoder.reset_profiler()
        start_time = time.time()
        total_tokens_spec = 0
        for prompt in prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            out = decoder.speculative_generate(inputs.input_ids, max_new_tokens=max_new_tokens, temperature=0.0)
            total_tokens_spec += (out.shape[1] - inputs.input_ids.shape[1])
            
        spec_time = time.time() - start_time
        spec_tps = total_tokens_spec / spec_time
        speedup = spec_tps / target_tps
        
        alpha = decoder.total_draft_tokens_accepted / max(1, decoder.total_draft_tokens_proposed)
        
        # Calculate theoretical optimal K using empirical alpha and cost ratio
        best_theoretical_k = 1
        best_theo_speedup = 0
        for test_k in range(1, 40):
            # E[tokens] = (1 - alpha^(k+1)) / (1 - alpha)
            e_tokens = (1 - (alpha ** (test_k + 1))) / (1 - alpha) if alpha < 1.0 else (test_k + 1)
            # Cost per step relative to target is 1 + k*c
            cost = 1 + test_k * c
            theo_speedup = e_tokens / cost
            if theo_speedup > best_theo_speedup:
                best_theo_speedup = theo_speedup
                best_theoretical_k = test_k
                
        print(f"| {k:<2} | {speedup:>6.2f}x | {spec_tps:>10.2f} | {alpha:>14.3f} | {best_theoretical_k:>10} |")

    print("\n--- Hardware Memory Profiling ---")
    if torch.cuda.is_available():
        target_mem = torch.cuda.max_memory_allocated(device) / (1024**3)
        print(f"Peak VRAM on Target GPU ({device}): {target_mem:.2f} GB")
        if str(device) != str(decoder.draft_device):
            draft_mem = torch.cuda.max_memory_allocated(decoder.draft_device) / (1024**3)
            print(f"Peak VRAM on Draft GPU ({decoder.draft_device}): {draft_mem:.2f} GB")

# 6. Execution Main
Run this cell to fire off the full pipeline!

In [ ]:
print("--- Phase 0: Scaffolding ---")
draft, target, tokenizer = load_models_and_tokenizer(
    draft_model_id="Qwen/Qwen2.5-0.5B-Instruct",
    target_model_id="Qwen/Qwen2.5-7B-Instruct",
)
device = next(target.parameters()).device
decoder = SpeculativeDecoder(draft, target, tokenizer, k=4)
max_new_tokens = 30
pass_count = 0
total = len(TEST_PROMPTS)

print(f"\n--- Phase 1 & 2: Correctness Test ({total} Prompts) ---")
print(f"Baseline: naive greedy (no logits processors)")
print(f"Test:     speculative decoder (k={decoder.k})")

for i, prompt in enumerate(TEST_PROMPTS):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs.input_ids

    baseline_out = naive_greedy_generate(
        target, tokenizer, input_ids, max_new_tokens
    )
    with torch.no_grad():
        spec_out = decoder.speculative_generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.0
        )

    min_len = min(baseline_out.shape[1], spec_out.shape[1])
    baseline_trunc = baseline_out[0, :min_len]
    spec_trunc = spec_out[0, :min_len]

    match = torch.equal(baseline_trunc, spec_trunc)

    if match:
        pass_count += 1
    else:
        print(f"Prompt {i+1}/{total} [FAIL]")
        break

print(f"\nResult: {pass_count}/{total} Passed.")
if pass_count == total:
    print("PHASE 2 COMPLETE: Speculative Decoder is 100% Greedy Exact-Match Correct!")
    
test_phase_3_sampling_math_proof(decoder)
test_phase_4_multi_gpu(decoder, target, tokenizer, device)
test_phase_5_benchmarking(decoder, target, draft, tokenizer, device)
